# Notebook 03: Tagging and Classification

**Objective:** Tag columns, classify PII data, and apply governance labels to catalog assets.

**Prerequisites:**
- Completed Notebooks 01 and 02 (lab running, tables discovered, descriptions added)
- Familiarity with the PATCH API from Notebook 02

**What you will learn:**
1. List available tag categories (classifications) in OpenMetadata
2. Tag the `email` column as PII
3. Tag the `phone` column as PII
4. Apply Tier classification to tables
5. Query all PII-tagged columns across the catalog

## Setup: Configuration and Authentication

In [ ]:
import requests
import json
import time

# OpenMetadata API base URL (internal Docker network)
OM_URL = "http://openmetadata-server:8585/api/v1"

def get_auth_token(base_url, username="admin", password="admin", max_retries=12, wait_seconds=15):
    """Authenticate with OpenMetadata and return JWT token."""
    login_url = f"{base_url}/users/login"
    payload = {"email": username, "password": password}
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(login_url, json=payload, timeout=10)
            if resp.status_code == 200:
                print(f"Authenticated successfully")
                return resp.json()["accessToken"]
            print(f"Attempt {attempt}/{max_retries}: HTTP {resp.status_code} - retrying...")
        except (requests.ConnectionError, requests.Timeout):
            print(f"Attempt {attempt}/{max_retries}: Server not ready - retrying...")
        time.sleep(wait_seconds)
    raise ConnectionError("Could not connect to OpenMetadata")

token = get_auth_token(OM_URL)
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
patch_headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json-patch+json"}

# Table FQN prefix
FQN_PREFIX = "ecommerce_postgres.ecommerce.public"

print("Ready for tagging exercises.")

## Exercise 1: List Available Classifications and Tags

OpenMetadata comes with built-in **classifications** (tag categories). Each classification contains multiple **tags**. Common built-in classifications include:

- **Tier**: Data importance levels (Tier1 = critical, Tier2 = important, etc.)
- **PII**: Personally Identifiable Information markers
- **PersonalData**: GDPR-related personal data categories

Let's explore what's available.

In [ ]:
# List all classifications (tag categories)
resp = requests.get(
    f"{OM_URL}/classifications",
    headers=headers,
    params={"limit": 50}
)

classifications = resp.json().get("data", [])
print(f"Available classifications: {len(classifications)}")
print("=" * 60)

for cls in classifications:
    name = cls.get("name", "")
    desc = cls.get("description", "No description")[:80]
    print(f"\n  Classification: {name}")
    print(f"  Description: {desc}")
    
    # Get tags within this classification
    tags_resp = requests.get(
        f"{OM_URL}/tags",
        headers=headers,
        params={"parent": name, "limit": 20}
    )
    tags = tags_resp.json().get("data", [])
    if tags:
        tag_names = [t.get("name", "") for t in tags]
        print(f"  Tags: {', '.join(tag_names)}")
    else:
        print(f"  Tags: (none found or use different endpoint)")

## Exercise 2: Tag the Email Column as PII

The `email` column in the `customers` table contains **Personally Identifiable Information (PII)**. Tagging it as such in the catalog:

1. Makes PII visible to data consumers and auditors
2. Enables automated access policies (e.g., mask PII for non-privileged users)
3. Supports GDPR/privacy compliance reporting

We use a PATCH request to add the PII tag to a specific column.

In [ ]:
# Get the customers table to find the email column index
fqn = f"{FQN_PREFIX}.customers"
resp = requests.get(f"{OM_URL}/tables/name/{fqn}", headers=headers, params={"fields": "columns,tags"})
table = resp.json()
columns = table.get("columns", [])

# Find the email column index
email_idx = None
for i, col in enumerate(columns):
    if col["name"] == "email":
        email_idx = i
        break

if email_idx is not None:
    print(f"Found 'email' column at index {email_idx}")
    
    # Add PII.Sensitive tag to the email column
    patch_payload = [
        {
            "op": "add",
            "path": f"/columns/{email_idx}/tags/0",
            "value": {
                "tagFQN": "PII.Sensitive",
                "source": "Classification",
                "labelType": "Manual"
            }
        }
    ]
    
    resp = requests.patch(f"{OM_URL}/tables/name/{fqn}", headers=patch_headers, json=patch_payload)
    
    if resp.status_code == 200:
        updated_col = resp.json()["columns"][email_idx]
        tags = [t["tagFQN"] for t in updated_col.get("tags", [])]
        print(f"Email column tags: {tags}")
        print("PII tag applied successfully!")
    else:
        print(f"Failed: {resp.status_code} - {resp.text[:200]}")
        print("\nNote: The exact tag FQN may vary by OM version.")
        print("Check Exercise 1 output for available PII tag names.")
else:
    print("Email column not found. Make sure metadata ingestion has run.")

## Exercise 3: Tag the Phone Column as PII

Apply the same PII classification to the `phone` column. Phone numbers are also PII under most privacy regulations (GDPR, CCPA, 152-FZ).

In [ ]:
# Re-fetch table to get current state (tags may affect column indices)
resp = requests.get(f"{OM_URL}/tables/name/{fqn}", headers=headers, params={"fields": "columns,tags"})
table = resp.json()
columns = table.get("columns", [])

# Find the phone column index
phone_idx = None
for i, col in enumerate(columns):
    if col["name"] == "phone":
        phone_idx = i
        break

if phone_idx is not None:
    print(f"Found 'phone' column at index {phone_idx}")
    
    patch_payload = [
        {
            "op": "add",
            "path": f"/columns/{phone_idx}/tags/0",
            "value": {
                "tagFQN": "PII.Sensitive",
                "source": "Classification",
                "labelType": "Manual"
            }
        }
    ]
    
    resp = requests.patch(f"{OM_URL}/tables/name/{fqn}", headers=patch_headers, json=patch_payload)
    
    if resp.status_code == 200:
        updated_col = resp.json()["columns"][phone_idx]
        tags = [t["tagFQN"] for t in updated_col.get("tags", [])]
        print(f"Phone column tags: {tags}")
        print("PII tag applied successfully!")
    else:
        print(f"Failed: {resp.status_code} - {resp.text[:200]}")
else:
    print("Phone column not found.")

## Exercise 4: Apply Tier Classification to Tables

**Tier classification** indicates data importance:

| Tier | Meaning | Example |
|------|---------|--------|
| Tier1 | Critical business data | customers, orders |
| Tier2 | Important operational data | products |
| Tier3 | Supporting/derived data | order_items |

Tier tags help prioritize governance efforts: Tier1 data gets the most rigorous quality checks, access controls, and documentation requirements.

In [ ]:
# Assign tier classifications to tables
tier_assignments = {
    "customers": "Tier.Tier1",
    "orders": "Tier.Tier1",
    "products": "Tier.Tier2",
    "order_items": "Tier.Tier3"
}

for table_name, tier_tag in tier_assignments.items():
    fqn = f"{FQN_PREFIX}.{table_name}"
    
    patch_payload = [
        {
            "op": "add",
            "path": "/tags/0",
            "value": {
                "tagFQN": tier_tag,
                "source": "Classification",
                "labelType": "Manual"
            }
        }
    ]
    
    resp = requests.patch(f"{OM_URL}/tables/name/{fqn}", headers=patch_headers, json=patch_payload)
    
    if resp.status_code == 200:
        tags = [t["tagFQN"] for t in resp.json().get("tags", [])]
        print(f"  {table_name}: {tags}")
    else:
        print(f"  {table_name}: failed ({resp.status_code}) - {resp.text[:100]}")

print("\nTier classification applied to all tables.")

## Exercise 5: Query All PII-Tagged Columns

A key governance capability: **find all PII columns across the entire catalog**. This is essential for:

- Privacy impact assessments
- Data subject access requests (GDPR Article 15)
- Access control policy enforcement
- Data masking configuration

In [ ]:
# Search for all assets tagged with PII
resp = requests.get(
    f"{OM_URL}/search/query",
    headers=headers,
    params={
        "q": "*",
        "index": "table_search_index",
        "query_filter": json.dumps({
            "query": {
                "bool": {
                    "must": [
                        {"term": {"tags.tagFQN": "PII.Sensitive"}}
                    ]
                }
            }
        }),
        "from": 0,
        "size": 50
    }
)

if resp.status_code == 200:
    hits = resp.json().get("hits", {}).get("hits", [])
    print(f"Tables with PII-tagged columns: {len(hits)}")
    print("=" * 60)
    for hit in hits:
        source = hit.get("_source", {})
        table_name = source.get("name", "")
        fqn = source.get("fullyQualifiedName", "")
        print(f"\nTable: {fqn}")
        
        # Find PII-tagged columns within this table
        for col in source.get("columns", []):
            col_tags = [t.get("tagFQN", "") for t in col.get("tags", [])]
            if any("PII" in tag for tag in col_tags):
                print(f"  Column: {col['name']} (type: {col.get('dataType', 'N/A')})")
                print(f"    Tags: {col_tags}")
else:
    print(f"Search returned: {resp.status_code}")
    print("Trying alternative approach...")

In [ ]:
# Alternative: Manually check each table for PII-tagged columns
print("Manual PII column scan:")
print("=" * 60)

pii_columns = []

for table_name in ["customers", "products", "orders", "order_items"]:
    fqn = f"{FQN_PREFIX}.{table_name}"
    resp = requests.get(f"{OM_URL}/tables/name/{fqn}", headers=headers, params={"fields": "columns,tags"})
    
    if resp.status_code == 200:
        table = resp.json()
        for col in table.get("columns", []):
            col_tags = [t.get("tagFQN", "") for t in col.get("tags", [])]
            if any("PII" in tag for tag in col_tags):
                pii_columns.append({
                    "table": table_name,
                    "column": col["name"],
                    "type": col.get("dataType", "N/A"),
                    "tags": col_tags
                })

print(f"\nFound {len(pii_columns)} PII-tagged columns:")
print(f"{'Table':<15} {'Column':<15} {'Type':<15} {'Tags'}")
print("-" * 60)
for col_info in pii_columns:
    print(f"{col_info['table']:<15} {col_info['column']:<15} {col_info['type']:<15} {col_info['tags']}")

print(f"\nPII Audit: {len(pii_columns)} columns require special handling (masking, access control, DSAR).")

## Summary

In this notebook, you learned:

1. **Classifications:** Built-in tag categories (PII, Tier, PersonalData) provide governance vocabulary
2. **Column tagging:** Mark specific columns as PII for privacy compliance
3. **Tier classification:** Prioritize governance effort by data importance level
4. **PII discovery:** Query the catalog to find all PII across the organization
5. **Manual vs. automated:** Tags can be applied manually (as here) or via automated classification rules

**Key concept:** Tagging transforms a catalog from a passive directory into an active governance tool. Tagged metadata enables automated policies, compliance reporting, and impact analysis.

**Next:** [04-search-lineage.ipynb](04-search-lineage.ipynb) - Full-text search and lineage exploration.